In [ ]:
# Switch notebook to runtime version 2026.07
!pip install git+https://github.com/uiuc-focal-lab/syncode.git

  Cloning https://github.com/uiuc-focal-lab/syncode.git to /tmp/pip-req-build-mpilcxof
  Running command git clone --filter=blob:none --quiet https://github.com/uiuc-focal-lab/syncode.git /tmp/pip-req-build-mpilcxof
  Resolved https://github.com/uiuc-focal-lab/syncode.git to commit 4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from syncode import Syncode

# Derived from syncode/parsers/grammars/calc.lark
grammar = """start: operands "," operator

    operands: NUMBER "," NUMBER

    operator: "+" | "-" | "*" | "/"

    %import common.NUMBER
    %import common.WS
    %ignore WS
"""

model_name = "microsoft/phi-2"

# Load the Syncode augmented model
syn_llm = Syncode(model=model_name, mode='grammar_mask', grammar=grammar)

[2026-10-22 21:29:06,609-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[2026-10-22 21:29:12,229-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-22 21:29:40,937-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_9057666741_50257.pkl exist: False
[2026-10-22 21:29:40,938-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_9057666741_50257.pkl.
[2026-10-22 21:29:41,483-syncode.mask_store.fsm_set] - 7 FSMs with 34 states initialized in 0.01 seconds
[2026-10-22 21:29:41,484-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-22 21:29:41,485-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 49 to 13


100%|██████████| 34/34 [00:08<00:00,  4.03it/s]


[2026-10-22 21:29:50,011-syncode.mask_store.mask_store] - Time taken to create mask store: 9.05 seconds


In [ ]:
def run_syncode(textual_query):
  prompt = f"""You are given a mathematical problem involving binary operations (addition, subtraction, multiplication, or division).
  Your task is to extract the two floating point operands and the (single) operator from the query.

  Examples:
  Query: What is 327. multiplied by 11.0?
  Output: 327.0, 11.0, *

  Query: What is 1.0001 plus .5?
  Output: 1.0001, 0.5, +

  Query: What is 120.4 divided by 4?
  Output: 120.4, 4.0, /

  Query: {textual_query}
  Output:
  """

  out = syn_llm.infer(prompt)[0]
  first, second, operator = out.split(",")
  first, second, operator = float(first.strip()), float(second.strip()), operator.strip()
  answer = None
  match operator:
    case "+":
      answer = first + second
    case "-":
      answer = first - second
    case "*":
      answer = first * second
    case "/":
      answer = first / second

  return ([first, second], operator, answer)

# Testing

In [ ]:
test_queries = [
    "What is 6.0 minus 2.5?",
    "What is 20. over 4.?",
    "What is 1000 times 3?",
    "What is .1 plus 1.000001?",
    "Subtract 4.0 from 15.0.",
    "Multiply 123.4 by 0.",
]

for query in test_queries:
  print(query, run_syncode(query))

What is 6.0 minus 2.5? ([6.0, 2.5], '-', 3.5)
What is 20. over 4.? ([20.0, 4.0], '/', 5.0)
What is 1000 times 3? ([1000.0, 3.0], '*', 3000.0)
What is .1 plus 1.000001? ([1.000001, 0.1], '+', 1.100001)
Subtract 4.0 from 15.0. ([15.0, 4.0], '-', 11.0)
Multiply 123.4 by 0. ([123.4, 0.0], '*', 0.0)
